# 3 - Generate Outcome Data

This notebook prepares gene expression outcome datasets from the LEAP spatial transcriptomics cohort so they can be used later for model training and evaluation (e.g., predicting gene expression from histology embeddings).

It loads GeoMx expression data and sample metadata, processes it, selects gene subsets, and exports cleaned outcome matrices for downstream modelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

def single_violin(expr,gene):
    # Create a violin plot of expression values
    plt.figure(figsize=(8, 6))
    sns.violinplot(y=expr, color="skyblue", inner="box")
    # Calculate statistics
    mean_expression = expr.mean()
    median_expression = expr.median()


    # Add mean and median lines
    plt.axhline(mean_expression, color="red",  linewidth=1)
    plt.axhline(median_expression, color="green",  linewidth=1)
    plt.text(0, mean_expression, f"Mean: {mean_expression:.2f}", color="red", fontsize=8)
    plt.text(0, median_expression, f"Median: {median_expression:.2f}", color="green", fontsize=8)

    # Customize the plot
    plt.title(f"Distribution of Expression Values for {gene}")
    plt.ylabel("Expression Value")
    plt.xlabel("All Samples (ROIs)")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from LEAPdata import get_slide_id, get_leap_ids

# Load the spatial data

In [ ]:
OME_LEVEL = 2 # should be level 1 but haven't extracted at 20x yet
DOWNSAMPLE = 2**OME_LEVEL
segment='CD45+'

root_path = "../data"
rawdata_path = os.path.join(root_path,'01-raw')
outcome_path = os.path.join(root_path,'02-input','outcomes')

In [ ]:

sample_metadata = pd.read_excel(os.path.join(rawdata_path,'metadata',"meta_data.xlsx"), index_col=0)
samples = pd.read_excel(os.path.join(rawdata_path,'metadata',"samples.xlsx"), index_col=0)

In [ ]:
samples.head()

In [ ]:
# Support metadata with DCC_ID either as a column or already as the index.
if 'DCC_ID' in sample_metadata.columns:
    sample_metadata = sample_metadata.set_index('DCC_ID')
sample_metadata.head()

In [ ]:
norm_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"norm_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

norm_expr_matrix = norm_matrix.astype(float)

In [ ]:
norm_expr_matrix.head()

In [ ]:
raw_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"raw_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

In [ ]:

expression_matrix = raw_matrix.astype(float)

In [ ]:
expression_matrix.head()

In [ ]:
# Replace '.' with '-' in column names
expression_matrix.columns = expression_matrix.columns.str.replace('.', '-', regex=False)
expression_matrix.columns = expression_matrix.columns.str.replace('-dcc', '.dcc', regex=False)
expression_matrix.columns
# Ensure data is in a usable format
#print(expression_matrix.head())  # Check expression matrix
#expression_matrix.columns

In [ ]:
expression_matrix.head()

## Only use data where we have both the gene expression and the metadata

keep only rows that are not pellets

In [ ]:
sample_metadata = sample_metadata[sample_metadata['Sample_Type'] != 'PELLET']

In [ ]:
rois_to_use = sorted(set(sample_metadata.index).intersection(set(expression_matrix.columns)))
len(rois_to_use)

In [ ]:
rois_to_use

In [ ]:
sample_metadata = sample_metadata.loc[rois_to_use].sort_index()
expression_matrix = expression_matrix[rois_to_use]
norm_expr_matrix = norm_expr_matrix[rois_to_use]

In [ ]:
sample_metadata.shape

# Identify hold out test set
need to group by patient


All segments from a patient must go into either train or test.

The test set must contain:

At least one CD45+ and one PanCK+ in Segment

At least one RESECTION and one DIAGNOSTIC in Sample_Type



In [ ]:
patient_segment_counts = sample_metadata.groupby(['Patient_ID', 'Segment']).size().unstack(fill_value=0)
patient_segment_counts

In [ ]:
patient_segment_counts[(patient_segment_counts['CD45+'] == 0) & 
                                         (patient_segment_counts['PanCK+'] == 0)]

In [ ]:
patient_summary = sample_metadata.groupby('Patient_ID').agg({
    'Segment': lambda x: set(x),
    'Sample_Type': lambda x: set(x)
})
patient_summary

In [ ]:
# Step 1: Get unique Patient_IDs
unique_patients = sample_metadata['Patient_ID'].unique()

# # Step 2: Randomly sample 10% of patients
# np.random.seed(42)
# sampled_patients = np.random.choice(unique_patients,
#                                      size=int(0.1 * len(unique_patients)),
#                                      replace=False)



In [ ]:
# # Step 3: Get all rows for those patients
# train_metadata = sample_metadata[ ~sample_metadata['Patient_ID'].isin(sampled_patients)] 
# test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(sampled_patients)]

these patients were chosen randomly using the code above and used to generate the  data but the seed was not set correctly so the above code changes every time 

In [ ]:
train_patients = {'GST15102',
 'GST17605',
 'GST17723',
 'GST18770',
 'GST18928',
 'GST18944',
 'GST19918',
 'GST23088',
 'GST23677',
 'GST24143',
 'GST25373',
 'GST25518',
 'GST26237',
 'GST26311',
 'GST26322',
 'GST26353',
 'GST26423',
 'GST26448',
 'GST26501',
 'GST26791',
 'GST26854',
 'GST26965',
 'GST27131',
 'GST27235',
 'GST27321',
 'GST27337',
 'GST27498',
 'GST27506',
 'GST27925',
 'GST29819',
 'GST30473',
 'GST35861',
 'GST36019',
 'GST36564',
 'GST36713',
 'GST36718',
 'GST36776',
 'GST36789',
 'GST37108',
 'GST37122',
 'GST37411',
 'GST37685',
 'GST37926',
 'GST41084',
 'GST41151',
 'GST43189',
 'GST43338',
 'GST43395',
 'GST43651',
 'GST43654',
 'GST43962',
 'GST44149',
 'GST44189',
 'GST44222',
 'GST44440',
 'GST44790',
 'GST44827',
 'GST46489',
 'GST46735',
 'GST46801',
 'GST47130',
 'GST47289',
 'GST47381',
 'GST47384',
 'GST47471',
 'GST47765',
 'GST47796',
 'GST47859',
 'GST47883',
 'GST48110',
 'GST48134',
 'GST48176',
 'GST48323',
 'GST48454',
 'GST48455',
 'GST48456',
 'GST48495',
 'GST48581',
 'GST48631',
 'GST48798',
 'GST49284'}
len(train_patients)

In [ ]:
for p in train_patients:
    print(p)

In [ ]:
test_patients = {'GST23703',
 'GST25222',
 'GST26416',
 'GST26519',
 'GST26889',
 'GST43294',
 'GST47380',
 'GST47661'}
len(test_patients)

In [ ]:
train_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(train_patients)]
test_metadata = sample_metadata[sample_metadata['Patient_ID'].isin(test_patients)]

check we have totally separated test and train

In [ ]:
(train_metadata.shape[0] + test_metadata.shape[0]) == sample_metadata.shape[0]

In [ ]:
train_metadata

In [ ]:
rois_train = sorted(set(train_metadata.index))
rois_test = sorted(set(test_metadata.index))

In [ ]:
print(f"train rois: {len(rois_train)}")
print(f"test rois: {len(rois_test)}")
print(f"{len(rois_test)/len(rois_train):%}")

In [ ]:
(len(rois_test)+len(rois_train))==len(rois_to_use)

In [ ]:
expression_train = expression_matrix[rois_train]
expression_test = expression_matrix[rois_test]

In [ ]:
print("Original Sample_Type proportions:")
print(sample_metadata['Sample_Type'].value_counts(normalize=True))

print("\nSampled Sample_Type proportions:")
print(train_metadata['Sample_Type'].value_counts(normalize=True))
print("\nSampled Sample_Type proportions:")
print(test_metadata['Sample_Type'].value_counts(normalize=True))

In [ ]:
expression_test.shape

In [ ]:
test_metadata.shape

# Gene Expression

In [ ]:
sample_metadata = sample_metadata.sort_index()
test_metadata = test_metadata.sort_index()
train_metadata = train_metadata.sort_index()

In [ ]:
rois_cd45_train = list(train_metadata[train_metadata['Segment']=='CD45+'].index)
rois_panck_train = list(train_metadata[train_metadata['Segment']=='PanCK+'].index)

In [ ]:
rois_cd45_test = list(test_metadata[test_metadata['Segment']=='CD45+'].index)
rois_panck_test = list(test_metadata[test_metadata['Segment']=='PanCK+'].index)

In [ ]:
log_transformed_expr = expression_matrix.apply(np.log1p).T

In [ ]:

#final_expression_matrix = expression_matrix.T.astype(float).sort_index()
#log_transformed_expr = final_expression_matrix.apply(np.log1p)
norm_expr_matrix = norm_expr_matrix.T.astype(float).sort_index()
norm_expr_matrix.head()

## Remove Contamination

Tumour segments have contamination from plasma cells so we need to remove these genes before doing our analysis

make sure we are only doing this for PanCK+ AOIs

In [ ]:
tumour_aois = sorted(rois_panck_test + rois_panck_train)

len(tumour_aois)

In [ ]:
plasma_genes = ["IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHM", "IGHA1", "IGKC", "IGLL5"]

In [ ]:
tumour_expr = norm_expr_matrix.loc[tumour_aois,:]
tumour_expr.head()
#tumour_expr.loc[plasma_genes,:].head()

check correlation of all genes with known plasma genes to identify other genes that are likely from plasma cells rather than tumour epithelial 

In [ ]:
# compute correlations of every gene vs. the mean plasma score
plasma_score = tumour_expr[plasma_genes].mean(axis=1)

corrs = tumour_expr.corrwith(plasma_score, axis=0)  # correlation of each gene with plasma_score
corrs = corrs.sort_values(ascending=False)

# turn into dataframe
corr_df = pd.DataFrame({
    "Gene": corrs.index,
    "Correlation_with_Plasma": corrs.values
})

Inspect correlation and remove anything over 0.7

In [ ]:
corr_df[corr_df['Correlation_with_Plasma']>0.7]

In [ ]:
contamination_genes = corr_df.loc[corr_df['Correlation_with_Plasma']>0.7,'Gene'].to_list()

we should consider anything over 0.5 as possible plasma - JCHAIN and MZB1 are definite plasma genes

In [ ]:
cond = (corr_df['Correlation_with_Plasma']<=0.7)&(corr_df['Correlation_with_Plasma']>0.5)
corr_df[cond]

In [ ]:
contamination_genes = contamination_genes + corr_df.loc[cond,'Gene'].to_list()
contamination_genes

In [ ]:
cond = (corr_df['Correlation_with_Plasma']<=0.5)&(corr_df['Correlation_with_Plasma']>0.4)
corr_df[cond]

Strongly immune-specific (should consider removing as we only want tumour-epithelial)

- CD79A (corr 0.46) → B-cell receptor, B lineage only.
- TRBC1 (corr 0.46) → T-cell receptor β chain, T lineage only.
- C1QA, C1QB, C1QC (corr 0.40, 0.45, 0.45) → complement genes, macrophage/myeloid lineage.
- ADA2 (corr 0.43) → myeloid/monocyte.
- LYZ (corr 0.48) → lysozyme, classic macrophage/neutrophil marker.
- IL10RA (corr 0.42) → monocyte/macrophage, not epithelial.
- EVI2B, CORO1A, WIPF1 (corr 0.42, 0.41, 0.41) → leukocyte cytoskeleton/trafficking, T/NK/myeloid associated.

## Training Set Only
Need to determine genes based on train set to avoid bias

In [ ]:
TRAIN_SET_ONLY = True

In [ ]:
final_norm_matrix = norm_expr_matrix.copy()

In [ ]:
if TRAIN_SET_ONLY:
    final_norm_matrix = final_norm_matrix.loc[rois_train]
    train_tumour_aois = set(tumour_aois).intersection(set(rois_train))
else:
    train_tumour_aois = tumour_aois

set expression to zero for contamination genes in tumour epithelial

In [ ]:
final_norm_matrix.loc[list(train_tumour_aois),contamination_genes] = 0
# check we have only set to 0 on the PanCK+ AOIs
final_norm_matrix.loc[:,contamination_genes]

## Setup AnnData object

use the normalised, batch-corrected data to find the HVGs and SVGs

In [ ]:
import scanpy as sc
import squidpy as sq
from anndata import AnnData

final_norm_matrix
sample_metadata

In [ ]:
if TRAIN_SET_ONLY:
    meta_to_use = train_metadata.sort_index().copy()
else:
    meta_to_use = sample_metadata.sort_index().copy()


In [ ]:
# final_norm_matrix = final_norm_matrix.sort_index().sort_index(axis=1)
# meta_to_use = meta_to_use.sort_index()



# adata.obsm["spatial"] = meta_to_use[["ROICoordinateX", "ROICoordinateY"]].to_numpy()
# adata.obs["LEAPID"] = meta_to_use["LEAP_ID"].astype(str).to_numpy()
# adata.obs["segment"] = meta_to_use["Segment"].astype(str).to_numpy()

# adata = adata[sorted(adata.obs_names), sorted(adata.var_names)].copy()

In [ ]:
# Create AnnData object
#adata = AnnData(final_norm_matrix) #expression matrix transposed
adata = AnnData(final_norm_matrix.loc[meta_to_use.index].copy())

# adata.obsm["spatial"] = sample_metadata[['ROICoordinateX', 'ROICoordinateY']].to_numpy()  #roi coords
# adata.obs["LEAPID"] = sample_metadata['LEAP_ID']
# adata.obs["segment"] = sample_metadata['Segment']

adata.obsm["spatial"] = meta_to_use[['ROICoordinateX', 'ROICoordinateY']].to_numpy()  #roi coords
adata.obs["LEAPID"] = meta_to_use['LEAP_ID']
adata.obs["segment"] = meta_to_use['Segment']

adata.X = np.nan_to_num(adata.X, nan=0.0, posinf=0.0, neginf=0.0)
adata = adata[sorted(adata.obs_names), sorted(adata.var_names)].copy()

## Highly Variable Genes (HVGs)

In [ ]:
from collections import defaultdict

# Dictionary to store HVG results per segment
hvg_results_by_segment = {}

# Loop over each segment type
for segment in ['CD45+','PanCK+']:
    print(f"Processing segment: {segment}")
    
    # Subset AnnData
    adata_sub = adata[adata.obs["segment"] == segment].copy()
    
    # Filter out genes with all-zero expression (optional but good)
    sc.pp.filter_genes(adata_sub, min_cells=1)
    
    # Identify highly variable genes
    sc.pp.highly_variable_genes(
        adata_sub,
        flavor='cell_ranger',
        n_top_genes=100
    )
    
    # Store DataFrame of HVGs
    hvg_df = adata_sub.var[adata_sub.var["highly_variable"]].copy()
    hvg_results_by_segment[segment] = hvg_df

    print(f"Found {hvg_df.shape[0]} highly variable genes for {segment}")


Get top 50 HVGs per segment

In [ ]:
HVGs_cd45 = list(hvg_results_by_segment['CD45+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_cd45

For tumour (PanCK+) segments we need to filter out any immune genes from plasma cells that have contaminated the segment, otherwise these will dominate the analyses

In [ ]:
hvg_results_by_segment['PanCK+'].index

Check none of the contamination genes made it into the HVGs

In [ ]:
hvg_results_by_segment['PanCK+'].index.isin(contamination_genes).any()

Check whether the borderline plasma-correlated genes are in our list of HVGs

In [ ]:
poss_genes = ['CD79A','TRBC1','C1QA','C1QB','C1QC','ADA2','LYZ','IL10RA','EVI2B','CORO1A','WIPF1']
hvg_results_by_segment['PanCK+'].index.isin(poss_genes).any()

In [ ]:
HVGs_panck = list(hvg_results_by_segment['PanCK+'].sort_values(by='dispersions_norm', ascending=False)[:50].index)
HVGs_panck

In [ ]:
from collections import defaultdict

top_n = 100
segment_to_top_genes = {
    segment: set(hvg_df.sort_values("dispersions_norm", ascending=False).index[:top_n])
    for segment, hvg_df in hvg_results_by_segment.items()
}

# Count how many segments each gene appears in
from collections import Counter

gene_counter = Counter()
for genes in segment_to_top_genes.values():
    gene_counter.update(genes)

# Show most common HVGs
print("Most frequently variable genes across segments:")
for gene, count in gene_counter.most_common(10):
    print(f"{gene}: {count} segments")

use the log transformed raw expression for the ground truth once we know what the HVGs are

In [ ]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_train,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [ ]:

###hvgs_df_panck = expression_matrix.T.loc[rois_panck,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
###hvgs_df_cd45 = expression_matrix.T.loc[rois_cd45,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [ ]:
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw.csv"), index=False)

TEST SET

In [ ]:
hvgs_df_panck = log_transformed_expr.loc[rois_panck_test,HVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,HVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
hvgs_df_panck.to_csv(os.path.join(outcome_path,f"hvg_panck_raw_TEST.csv"), index=False)
hvgs_df_cd45.to_csv(os.path.join(outcome_path,f"hvg_cd45_raw_TEST.csv"), index=False)

## Spatially Variable Genes (SVGs)

In [ ]:
### ORIGINAL CODE

# results = {}

# for leapid in adata.obs["LEAPID"].unique():
#     for segment in ['CD45+','PanCK+']:
#         print(segment)
#         subset = adata[
#             (adata.obs["LEAPID"] == leapid) &
#             (adata.obs["segment"] == segment)
#         ].copy()
        

#         # Compute spatial graph and spatial autocorrelation
#         n_obs = subset.n_obs


#         if n_obs > 1:
#             print(leapid, segment, n_obs)
#             #print(f"{leapid} {segment} {n_obs}")
#             n_neighbors = min(5, n_obs - 1)


#             # Build spatial graph and compute Moran’s I
#             sq.gr.spatial_neighbors(subset, coord_type="generic", n_neighs=n_neighbors)
#             sq.gr.spatial_autocorr(subset, mode="moran")

#             # Extract top genes
#             moranI = subset.uns['moranI']
#             top_genes = moranI.sort_values("I", ascending=False).head(10)
            
#             results[(leapid, segment)] = top_genes
#             #print(f"Top genes for {leapid}-{segment}: {top_genes.index.tolist()}")

In [ ]:


# # Dictionary: segment → Counter of gene frequencies
# gene_counts_by_segment = defaultdict(Counter)

# top_n = 20  # Adjust based on how many top genes you consider

# for (leapid, segment), df in results.items():
#     top_genes = df.head(top_n).index.tolist()
#     gene_counts_by_segment[segment].update(top_genes)

LEAP115
LEAP130
LEAP012
LEAP106
LEAP028

In [ ]:
# test_leapid = "LEAP012"
# test_segment = "CD45+"
# print(f"{test_leapid} run 1")
# subset = adata[
#     (adata.obs["LEAPID"] == test_leapid) &
#     (adata.obs["segment"] == test_segment)
# ].copy()

# sq.gr.spatial_neighbors(
#     subset,
#     coord_type="generic",
#     n_neighs=min(5, subset.n_obs - 1)
# )

# sq.gr.spatial_autocorr(
#     subset,
#     mode="moran",
#     n_perms=None,
#     n_jobs=1,
#     seed=42
# )

# subset.uns["moranI"].sort_values("I", ascending=False).head(20)

In [ ]:

# print(f"{test_leapid} run 2")
# subset = adata[
#     (adata.obs["LEAPID"] == test_leapid) &
#     (adata.obs["segment"] == test_segment)
# ].copy()

# sq.gr.spatial_neighbors(
#     subset,
#     coord_type="generic",
#     n_neighs=min(5, subset.n_obs - 1)
# )

# sq.gr.spatial_autocorr(
#     subset,
#     mode="moran",
#     n_perms=None,
#     n_jobs=1,
#     seed=42
# )

# subset.uns["moranI"].sort_values("I", ascending=False).head(20)

### main loop

In [ ]:
import random
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
results = {}
segments = ["CD45+", "PanCK+"]

# Calculate Moran's I within each LEAPID × segment
for leapid in adata.obs["LEAPID"].unique():
    for segment in segments:
        
        subset = adata[
            (adata.obs["LEAPID"] == leapid) &
            (adata.obs["segment"] == segment)
        ].copy()

        # Force deterministic observation order
        subset = subset[sorted(subset.obs_names)].copy()

        n_obs = subset.n_obs
        # Skip tiny or degenerate spatial graphs
        if n_obs < 6:
            print("skipping ",leapid)
            continue


        print(leapid, segment, n_obs)

        n_neighbors = min(5, n_obs - 1)
        print("1")
        sq.gr.spatial_neighbors(
            subset,
            coord_type="generic",
            n_neighs=n_neighbors
        )


        sq.gr.spatial_autocorr(
            subset,
            mode="moran",
            n_perms=None,
            n_jobs=1,
            seed=SEED
        )
        moranI = subset.uns["moranI"].copy()

        moranI = moranI.dropna(subset=["I"])

        if "var_norm" in moranI.columns:
            moranI = moranI[moranI["var_norm"] > 0]

        # Keep positive spatial autocorrelation only
        moranI = moranI[moranI["I"] > 0]

        # Deterministic ranking with gene-name tie break
        moranI = (
            moranI
            .reset_index(names="gene")
            .sort_values(
                ["I", "gene"],
                ascending=[False, True],
                kind="mergesort"
            )
            .set_index("gene")
        )

        results[(leapid, segment)] = moranI


In [ ]:
print(len(results))
print(sorted(results.keys())[:20])

### Count

In [ ]:
combined_df = pd.concat(
    [
        df.assign(LEAPID=leapid, segment=segment)
        for (leapid, segment), df in results.items()
    ]
).reset_index(names="gene")

In [ ]:
combined_df

In [ ]:
#combined_df.to_csv(os.path.join(outcome_path,"svg_summary.csv"), index=False)

In [ ]:
from collections import defaultdict, Counter

# Aggregate recurrent SVGs across LEAPIDs
top_n_per_leapid = 20


gene_scores_by_segment = defaultdict(list)

for (leapid, segment), moranI in results.items():
    top = moranI.head(top_n_per_leapid)

    for gene, row in top.iterrows():
        gene_scores_by_segment[segment].append({
            "gene": gene,
            "LEAPID": leapid,
            "I": row["I"]
        })


In [ ]:
min_frequency = 2
min_mean_I = 0.1
n_final_genes = 50

svg_summary = {}

for segment, records in gene_scores_by_segment.items():
    tmp = pd.DataFrame(records)

    summary = (
        tmp.groupby("gene")
        .agg(
            frequency=("gene", "size"),
            mean_I=("I", "mean"),
            median_I=("I", "median"),
            max_I=("I", "max")
        )
    )

    summary = summary[
        (summary["frequency"] >= min_frequency) &
        (summary["mean_I"] > min_mean_I) &
        (summary["median_I"] > 0)
    ]

    summary = summary.sort_values(
        ["frequency", "median_I", "mean_I", "max_I"],
        ascending=[False, False, False, False],
        kind="mergesort"
    )

    svg_summary[segment] = summary


SVGs_cd45 = svg_summary["CD45+"].head(n_final_genes).index.tolist()
SVGs_panck = svg_summary["PanCK+"].head(n_final_genes).index.tolist()

Get top 50 SVGs per segment

In [ ]:
# SVGs_cd45 = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='CD45+']
# SVGs_cd45
# SVGs_panck = [gene for segment, counter in gene_counts_by_segment.items() for gene,_ in counter.most_common(50) if segment=='PanCK+']
# SVGs_panck

In [ ]:
n_final_genes = 50
SVGs_cd45 = svg_summary["CD45+"].head(n_final_genes).index.tolist()
SVGs_panck = svg_summary["PanCK+"].head(n_final_genes).index.tolist()


In [ ]:
svg_summary["CD45+"].head(n_final_genes)

In [ ]:
svg_summary["PanCK+"].head(n_final_genes)

In [ ]:
#SVGs_cd45_run1  = SVGs_cd45.copy()
#SVGs_panck_run1  = SVGs_panck.copy()

In [ ]:
# sorted(SVGs_cd45_run1) == sorted(SVGs_cd45)
# sorted(SVGs_panck_run1) == sorted(SVGs_panck)

In [ ]:
SVGs_cd45 = sorted(SVGs_cd45)
SVGs_cd45

In [ ]:
SVGs_panck = sorted(SVGs_panck)
SVGs_panck

Check whether any of the contamination genes are in the SVGs

In [ ]:
# SVGs_cd45 = ["MGP","RPS17","PTBP1","FN1","POSTN","LAPTM4B","RPS19","RPS4X","RPL32","RPS12","HSPA8","XAF1","CLU","COTL1","COL3A1","LGALS1","AEBP1","BST2","GPNMB","IFI44","IFI44L","EIF4A2","BGN","FDCSP"]
# SVGs_panck = ["CRABP2","HLA-DPA1","MAZ","RPS19","TM4SF1","TUBB4B","LGALS1","MZT2B","ELF5","CTHRC1","IFI6","IFITM1","CD44","S100A7","TMSB4X","RPS5","SCARA3","RPS3A","RPL35","RPS6","MMP7","HLA-E","FBXO32","CHI3L1","C3","MYL12B"]
# SVGs_cd45 = sorted(SVGs_cd45)
# SVGs_panck = sorted(SVGs_panck)

In [ ]:
set(SVGs_panck).intersection(set(contamination_genes))

In [ ]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_train,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_train,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [ ]:
#svgs_df_cd45[new_svg_cd45]

In [ ]:

####svgs_df_panck = expression_matrix.T.loc[rois_panck,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
####svgs_df_cd45 = expression_matrix.T.loc[rois_cd45,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})

In [ ]:
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw.csv"), index=False)

TEST SET

In [ ]:
svgs_df_panck = log_transformed_expr.loc[rois_panck_test,SVGs_panck].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_cd45 = log_transformed_expr.loc[rois_cd45_test,SVGs_cd45].reset_index().rename(columns={"index": "ROI_ID"})
svgs_df_panck.to_csv(os.path.join(outcome_path,f"svg_panck_raw_TEST.csv"), index=False)
svgs_df_cd45.to_csv(os.path.join(outcome_path,f"svg_cd45_raw_TEST.csv"), index=False)

Check overlap between SVGs and HVGs

In [ ]:
set(SVGs_panck).intersection(set(HVGs_panck))

In [ ]:
set(SVGs_cd45).intersection(set(HVGs_cd45))

In [ ]:
#old_hvgs = HVGs_cd45 + HVGs_panck

In [ ]:
#old_svgs = SVGs_cd45 + SVGs_panck

In [ ]:
# new_hvgs = HVGs_cd45 + HVGs_panck
# new_svgs = SVGs_cd45 + SVGs_panck
# len(set(old_hvgs).intersection(set(new_hvgs)))
#len(set(old_svgs).intersection(set(new_svgs)))

## Gene Panels

In [ ]:
gene_panel_cd45 = ['CD274','CXCL9','IFNG','BCL2','CD3D', 'CD8A', 'CD4', 'FOXP3', 'CD68', 'CD163','IDO1', 'LAG3', 'STAT1', 'S100A9']
gene_panel_panck = ['BRCA1','BRCA2', 'TP53', 'PIK3CA','MYC','EGFR', 'AR', 'PTEN', 'RB1', 'CD274','BCL2','KRT5', 'KRT14', 'TP63', 'MKI67', 'VIM', 'MMP9','SOX10', 'ALDH1A1', 'CD44', 'ZEB1', 'SNAI1', 'CCNB1']

panel_df_panck = log_transformed_expr.loc[rois_panck_train,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_train,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})

####panel_df_panck = expression_matrix.T.loc[rois_panck,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
####panel_df_cd45 = expression_matrix.T.loc[rois_cd45,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"panel_cd45_raw.csv"), index=False)

# TEST SET
panel_df_panck = log_transformed_expr.loc[rois_panck_test,gene_panel_panck].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_cd45 = log_transformed_expr.loc[rois_cd45_test,gene_panel_cd45].reset_index().rename(columns={"index": "ROI_ID"})
panel_df_panck.to_csv(os.path.join(outcome_path,f"panel_panck_raw_TEST.csv"), index=False)
panel_df_cd45.to_csv(os.path.join(outcome_path,f"panel_cd45_raw_TEST.csv"), index=False)

# Check overlap of genes

In [ ]:
table_path = os.path.join(root_path, "..", "paper", "01 revisions", "tables")
os.makedirs(table_path, exist_ok=True)

In [ ]:
df_panel = pd.DataFrame(data = panel_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_panck.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_panck = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_panck['Segment'] = 'PanCK+'
df_panck.shape

In [ ]:
duplicated_panck = list(df_panck[df_panck.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_panck)} genes are duplicated in panck")
print(duplicated_panck)
df_panck[df_panck['Gene'].isin(duplicated_panck)].sort_values(['Gene'])

In [ ]:
unique_panck = list(df_panck['Gene'].unique())
print(f"Total genes (PanCK): {df_panck.shape[0]}")
print(f"Duplicate genes (PanCK): {len(duplicated_panck)} ")
print(f"Unique genes (PanCK): {len(unique_panck)}")

In [ ]:
df_panel = pd.DataFrame(data = panel_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_panel['Geneset'] = 'Panel'
df_hvg = pd.DataFrame(data = hvgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_hvg['Geneset'] = 'HVG'
df_svg = pd.DataFrame(data = svgs_df_cd45.drop('ROI_ID',axis=1).columns,columns=['Gene'])
df_svg['Geneset'] = 'SVG'
df_cd45 = pd.concat([df_panel,df_hvg,df_svg]).reset_index(drop=True)
df_cd45['Segment'] = 'CD45+'
df_cd45.shape

In [ ]:
duplicated_cd45 = list(df_cd45[df_cd45.duplicated('Gene')].Gene.unique())
print(f"{len(duplicated_cd45)} genes are duplicated in cd45")
print(duplicated_cd45)
df_cd45[df_cd45['Gene'].isin(duplicated_cd45)].sort_values(['Gene'])

In [ ]:
unique_cd45 = list(df_cd45['Gene'].unique())


df_cd45.shape

In [ ]:

print(f"Total genes (CD45): {df_cd45.shape[0]}")
print(f"Duplicate genes (CD45): {len(duplicated_cd45)} ")
print(f"Unique genes (CD45): {len(unique_cd45)}")

In [ ]:
print(f"CD45+ # unique genes: {len(unique_cd45)}")
print(f"PanCK+ # unique genes: {len(unique_panck)}")


In [ ]:
print(f"# genes across both segments: {len(set(unique_panck).union(set(unique_cd45)))}")

In [ ]:
df_summary = pd.DataFrame({
    "Compartment": ["CD45", "PanCK"],
    "Total genes": [df_cd45.shape[0], df_panck.shape[0]],
    "Duplicated genes": [len(duplicated_cd45), len(duplicated_panck)],
    "Unique genes": [len(unique_cd45), len(unique_panck)],
    "Specific genes": [
        len(set(unique_cd45).difference(set(unique_panck))),
        len(set(unique_panck).difference(set(unique_cd45))),
    ],
    "inboth": [
        len(set(unique_panck).intersection(set(unique_cd45))),
        len(set(unique_panck).intersection(set(unique_cd45)))],
})

df_summary.loc[len(df_summary)] = [
    "Total",
    df_summary["Total genes"].sum(),
    df_summary["Duplicated genes"].sum(),
    df_summary["Unique genes"].sum(),
    df_summary["Specific genes"].sum(),
    df_summary["inboth"].sum(),
]

In [ ]:
df_summary

In [ ]:
df_summary.to_csv(os.path.join(table_path,'ST1.csv'), index=False)

In [ ]:
print(f"# genes in both segments: {len(set(unique_panck).intersection(set(unique_cd45)))}")

In [ ]:
print(f"# genes only in CD45+: {len(set(unique_cd45).difference(set(unique_panck)))}")
print(f"# genes only in PanCK+: {len(set(unique_panck).difference(set(unique_cd45)))}")


In [ ]:
print(f"genes only in CD45+: {(set(unique_cd45).difference(set(unique_panck)))}")
print(f"genes only in PanCK+: {(set(unique_panck).difference(set(unique_cd45)))}")


# Save Standard Deviation
useful for calculating nRMSE

In [ ]:
print(len(set(gene_panel_panck + SVGs_panck )))
print(len(gene_panel_panck) + len(SVGs_panck))
print(set(gene_panel_panck).intersection(set(SVGs_panck)))

In [ ]:
allgenes_panck = list(set(gene_panel_panck + HVGs_panck + SVGs_panck))
allgenes_cd45 = list(set(gene_panel_cd45 + HVGs_cd45 + SVGs_cd45))

In [ ]:
extra_cd45 = ['CALML5', 'CXCL14']
extra_panck = ['ANKRD30A', 'FOXA1', 'IGF2', 'MSLN', 'TFF3']
allgenes_panck = allgenes_panck + extra_panck
allgenes_cd45 = allgenes_cd45 + extra_cd45

In [ ]:
std_panck_test = pd.DataFrame(log_transformed_expr.loc[rois_panck_test,allgenes_panck].std(axis=0),columns=['std'])
std_cd45_test = pd.DataFrame(log_transformed_expr.loc[rois_cd45_test,allgenes_cd45].std(axis=0),columns=['std'])
std_panck_test.index.name='Gene'
std_cd45_test.index.name='Gene'
std_panck_test.to_csv(os.path.join(outcome_path,f"allgenes_panck_test_std.csv"), index=True)
std_cd45_test.to_csv(os.path.join(outcome_path,f"allgenes_cd45_test_std.csv"), index=True)

# Additional

I want to know which leap ids have high variance on the genes of interest

CD45:
Panel ['CXCL9', 'CD68', 'STAT1', 'S100A9']
['TENT5C',
 'CCSAP',
 'ST6GALNAC3',
 'UBE2E1',
 'CD82',
 'DMRTC1B']
PanCK:
Panel ['PIK3CA', 'EGFR', 'MKI67']
['MKI67',
 'PIK3CA',
 'CA9',
 'LCN2',
 'AQP3',
 'IGHM']

 genes_of_interest[('CD45+','Panel')] = ['BCL2', 'IDO1', 'STAT1', 'CD274', 'CD3D', 'CXCL9'] #Panel
genes_of_interest[('CD45+','HVG')] = ['AREG','FDCSP','CPB1']
genes_of_interest[('CD45+','SVG')] = ['ST6GALNAC3','PRMT8','C7','TARP', 'TENT5C','RPL41','P2RY12','PITX1','CCSAP' ] #/SVG
genes_of_interest[('PanCK+','Panel')] = ['BRCA1', 'SNAI1', 'PIK3CA', 'BRCA2', 'EGFR', 'SNAI1','PIK3CA', 'BRCA2','EGFR'] #Panel
genes_of_interest[('PanCK+','HVG')] = ['IGHG1', 'IGHG4', 'IGHG3', 'IGKC',  'IGHG2',   'CSN3', 'CA9', 'CPB1' ]
genes_of_interest[('PanCK+','SVG')] = ['ATP5MC2', 'BRINP3', 'DMRTB1', 'NWD2', 'ENO1']

In [ ]:
cd45_genes_of_interest = ['CXCL9', 'CD68', 'STAT1', 'S100A9','TENT5C','CCSAP', 'ST6GALNAC3', 'UBE2E1', 'CD82', 'DMRTC1B']
panck_genes_of_interest = ['PIK3CA', 'EGFR', 'MKI67', 'CA9', 'LCN2', 'AQP3', 'IGHM']

In [ ]:
cd45_genes_of_interest = ['BCL2', 'IDO1', 'STAT1', 'CD274','AREG','FDCSP','CPB1','ST6GALNAC3','PRMT8','C7','TARP']
panck_genes_of_interest = ['BRCA1', 'SNAI1', 'PIK3CA', 'IGHG1', 'IGHG4', 'IGKC','ATP5MC2', 'BRINP3', 'DMRTB1','BRCA2', 'EGFR']

In [ ]:
expr_of_interest_panck = log_transformed_expr.loc[rois_panck_test,panck_genes_of_interest]
expr_of_interest_cd45 = log_transformed_expr.loc[rois_cd45_test,cd45_genes_of_interest]

In [ ]:
expr_of_interest_cd45 = sample_metadata.loc[rois_cd45_test,['LEAP_ID']].join(expr_of_interest_cd45,how='left')
expr_of_interest_panck = sample_metadata.loc[rois_panck_test,['LEAP_ID']].join(expr_of_interest_panck,how='left')


In [ ]:
for gene in cd45_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_cd45.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

In [ ]:
for gene in panck_genes_of_interest:
    print(f"{gene}     \t{list(expr_of_interest_panck.groupby('LEAP_ID').std().sort_values(gene, ascending=False).index[:5])}")

best for CD45
LEAP120
LEAP079
LEAP121

best for PanCK
LEAP121
LEAP065
LEAP082
LEAP120

In [ ]:
log_transformed_expr.loc[rois_cd45_test,allgenes_cd45]

# END
